# Blend Farm — Colab Worker

This notebook turns one temporary Google Colab GPU runtime into one Blend Farm worker. Before continuing:

1. Select **Runtime → Change runtime type → GPU**.
2. Start the central server and confirm its public `/login` page works.
3. In the dashboard, click **Enroll** to create a fresh one-time code.

Run every cell in order. The final cell stays active while the worker is rendering. A second node requires a separate Colab runtime and a separate enrollment code. Colab runtimes are temporary, so reconnecting to a new runtime requires enrolling it again.

In [ ]:
# @title 1. Worker settings
REPOSITORY_URL = "https://github.com/superintendent2521/render-farm.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
SERVER_URL = "https://farm.superintendent.me"  # @param {type:"string"}
WORKER_NAME = "colab-worker-1"  # @param {type:"string"}
DEVICE = "OPTIX"  # @param ["OPTIX", "CUDA", "AUTO", "CPU"]
CACHE_GB = 20  # @param {type:"integer"}

SERVER_URL = SERVER_URL.rstrip("/")
assert SERVER_URL.startswith("https://"), "SERVER_URL must use HTTPS"
assert WORKER_NAME.strip(), "WORKER_NAME cannot be empty"
assert DEVICE in {"OPTIX", "CUDA", "AUTO", "CPU"}
assert CACHE_GB >= 5, "Allow at least 5 GB for Blender and project files"
print(f"Worker: {WORKER_NAME} → {SERVER_URL} ({DEVICE})")

In [ ]:
# @title 2. Verify the assigned GPU
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU is attached. Choose Runtime → Change runtime type → GPU, then reconnect.")
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
# @title 3. Install or update the Blend Farm worker
import subprocess
import sys

package = f"git+{REPOSITORY_URL}@{BRANCH}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", package])
print("Blend Farm worker installed.")

In [ ]:
# @title 4. Test the central server
import httpx

response = httpx.get(f"{SERVER_URL}/healthz", timeout=30, follow_redirects=True)
response.raise_for_status()
health = response.json()
assert health.get("ok") is True, health
print(f"Central server is reachable: {health}")

## Enroll this runtime

Generate a new code in the dashboard immediately before running the next cell. The code is requested through a hidden prompt so it is not stored in the notebook. Enrollment codes are single-use and expire after ten minutes.

In [ ]:
# @title 5. Enroll securely
from getpass import getpass
from renderfarm.worker import CONFIG_FILE

if CONFIG_FILE.exists():
    print(f"This runtime is already enrolled ({CONFIG_FILE}). Skipping enrollment.")
else:
    enrollment_code = getpass("One-time enrollment code: " ).strip()
    if not enrollment_code:
        raise RuntimeError("An enrollment code is required")
    command = [
        sys.executable, "-m", "renderfarm.worker", "enroll",
        "--server", SERVER_URL,
        "--code", enrollment_code,
        "--name", WORKER_NAME,
        "--device", DEVICE,
        "--cache-gb", str(CACHE_GB),
    ]
    subprocess.check_call(command)
    del enrollment_code
    print("Enrollment complete.")

In [ ]:
# @title 6. Run diagnostics and install the required Blender version
# The first run downloads an official portable Blender build and may take several minutes.
subprocess.check_call([sys.executable, "-u", "-m", "renderfarm.worker", "doctor"] )

In [ ]:
# @title 7. Start rendering (leave this cell running)
print("Starting worker. Stop this cell to disconnect cleanly.")
try:
    subprocess.run([sys.executable, "-u", "-m", "renderfarm.worker", "run"], check=False)
except KeyboardInterrupt:
    print("Worker stopped.")

## Troubleshooting

- **No GPU:** reconnect after selecting a GPU runtime.
- **OptiX initialization fails:** change `DEVICE` to `CUDA`, then use a fresh runtime and enrollment code.
- **Enrollment rejected:** generate a new code; codes expire and can be used only once.
- **Worker disappears:** Colab may reclaim or disconnect the runtime. Its active frame returns to the queue after the lease expires.
- **Second worker:** open another copy of this notebook in a separate runtime, change `WORKER_NAME`, and generate another enrollment code.
- **Code changes are missing:** commit and push the local repository before rerunning the installation cell.